# Linac Optimaiztion Using the Bayesian Gaussian Process

## python environment and modules loading

In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scipy.constants as spc
import subprocess
import os, re
from skopt import gp_minimize
from skopt.space import Real
from skopt.plots import plot_convergence, plot_objective, plot_evaluations, plot_gaussian_process
from functools import partial
import pandas as pd
import pickle
import multiprocessing as mp

In [3]:
# Nicer plotting
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12,8)
%config InlineBackend.figure_format = 'retina'

## Astra binary enviroment loading

In [4]:
from astra import Astra
# load astra and generator binaries
%env ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
%env GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
!echo $ASTRA_BIN
!echo $GENERATOR_BIN

env: ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
env: GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator


In [5]:
from run_astra import *
from mobo_utils import *
from utils import *
from file_io import *
from plot_utils import *

/home/cspark/Work/simulation_codes-working/miniforge3/envs/linac-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# initial parameters extracted from the previous moga runs
init_parameters = [0.1947739, 
                   1.28652859, 
                   -2.88668503, 
                   35.62497684, 
                   -39.50903029, 
                   310.0534192]

In [7]:
T = 200e6
gamma = get_gamma(T)
beta = get_beta(gamma)
print (beta, gamma, beta* gamma)

0.999996752610937 392.3902367147747 392.38896247101155


In [8]:
# ratio from base parameters and lower and upper bounds
ratio = 0.50
lb = 1 - ratio
ub = 1 + ratio

# Define the search space for the optimization variables
search_space = [
    Real(lb*init_parameters[0], ub*init_parameters[0], name='solenoid_strength'),              # Solenoid strength
    Real(lb*init_parameters[1], ub*init_parameters[1], name='quad_focus_strength'),            # Focusing quadrupole strength
    Real(ub*init_parameters[2], lb*init_parameters[2], name='quad_defocus_strength'),          # Defocusing quadrupole strength
    Real(lb*init_parameters[3], ub*init_parameters[3], name='gun_cavity_phase'),          # Input phase of the RF gun cavity
    Real(ub*init_parameters[4], lb*init_parameters[4], name='acc12_cavity_phase'),        # Input phases of the ACC 1&2 cavities
    Real(lb*init_parameters[5], ub*init_parameters[5], name='acc34_cavity_phase') 
]

In [9]:
weight_combinations = [
    [0.05, 0.05, 0.9],
    [0.1, 0.1, 0.8],
    [0.15, 0.15, 0.7],
    [0.2, 0.2, 0.6],
    [0.25, 0.25, 0.5],
    [0.3, 0.3, 0.4],
    [0.35, 0.35, 0.3],
    [0.4, 0.4, 0.2],
    [0.45, 0.45, 0.1],
]

In [ ]:
# Run Bayesian Optimization
all_results = {}

n_calls = 150
n_jobs = 12

for weights in weight_combinations:
    weights_tuple = tuple(weights) # Convert to a tuple to use as a dictionary key
    print(f"Running optimization for weights: {weights}\n")

    # Use `partial` to create a new objective function that already has the 'weights' argument set.
    # gp_minimize will only see the 'parameters' argument.
    new_objective_func = partial(get_weighted_objective, weights=weights)

    # Run Bayesian Optimization
    result = gp_minimize(
        func=new_objective_func,
        dimensions=search_space,
        base_estimator="GP",
        acq_func="EI",
        n_calls=n_calls,
        n_jobs=n_jobs,
        random_state=56,
        verbose=False
    )

    # Store the results
    all_results[weights_tuple] = {
        'best_parameters': result.x,
        'best_objective_value': result.fun,
        'all_evaluations': result.func_vals
    }
    
    # Create a unique filename for this set of weights
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    
    # Save the result object to a file using pickle
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'
    with open(pickle_filename, 'wb') as f:
        pickle.dump(result, f)

    # Store results in a DataFrame
    df_results = pd.DataFrame({
        'Solenoid Strength': [x[0] for x in result.x_iters],
        'Focusing Quad Strength': [x[1] for x in result.x_iters],
        'Defocusing Quad Strength': [x[2] for x in result.x_iters],
        'RF Gun Cavity Phase': [x[3] for x in result.x_iters],
        'ACC1&2 Cavity Phase': [x[4] for x in result.x_iters],
        'ACC3&4 Cavity Phase': [x[5] for x in result.x_iters],
        'Objective (Sum of Emittance and Energy Spread)': result.func_vals
    })

    # Save results to a CSV file
    csv_filename = f'data/optimization_results_{filename_weights}.csv'
    df_results.to_csv(csv_filename, index=False)

    print(f"Best parameters:\n {result.x},\n Best value: {result.fun:.4e}\n")

Running optimization for weights: [0.05, 0.05, 0.9]

Running simulation with parameters: [0.11607360224014303, 1.7134224379081044, -3.1378254762632123, 39.16986068586428, -52.082758059275, 238.32272611736607]


In [ ]:
num_processes = 12 #mp.cpu_count()
print(f"Running simulations on {num_processes} cores...")

for weights in weight_combinations:
    print(f"Running optimization for weights: {weights}\n")
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'
    csv_filename = f'data/optimization_results_{filename_weights}.csv'
    
    with open(pickle_filename, 'rb') as f:
        result = pickle.load(f)
    
    df_results = pd.read_csv(csv_filename)
    
    with mp.Pool(num_processes) as pool:
        # Convert the DataFrame rows to a list of lists for the pool.map function
        parameter_list_of_lists = df_results.values.tolist()
        
        # Use pool.map to apply the run_simulation function to each list of parameters
        # The results will be gathered in the order of the input list
        all_output_list = pool.map(run_astra_simulation, parameter_list_of_lists)

    print(f"Simultions Done")
    
    output_filename = f'results/output_{filename_weights}.pkl'
    with open(output_filename, 'wb') as f:
        pickle.dump(all_output_list, f)

## For specific weights

In [ ]:
n = 3

weights = weight_combinations[n]
filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'
csv_filename = f'data/optimization_results_{filename_weights}.csv'

output_filename = f'results/output_{filename_weights}.pkl'

with open(pickle_filename, 'rb') as f:
    result = pickle.load(f)
    
df_results = pd.read_csv(csv_filename)

min_index = np.argmin(result.func_vals)
print(min_index, result.func_vals[min_index])

In [ ]:
with open(output_filename, 'rb') as f:
    loaded_output_list = pickle.load(f)

In [ ]:
norm_emit_x = [stats['norm_emit_x'][-1] for stats in loaded_output_list]
norm_emit_y = [stats['norm_emit_y'][-1] for stats in loaded_output_list]
sigma_energy = [stats['sigma_energy'][-1] for stats in loaded_output_list]

sigma_x = [stats['sigma_x'][-1] for stats in loaded_output_list]
sigma_xp = [stats['sigma_xp'][-1] for stats in loaded_output_list]
sigma_y = [stats['sigma_y'][-1] for stats in loaded_output_list]
sigma_yp = [stats['sigma_yp'][-1] for stats in loaded_output_list]
sigma_z = [stats['sigma_z'][-1] for stats in loaded_output_list]
mean_kinetic_energy = [stats['mean_kinetic_energy'][-1] for stats in loaded_output_list]

In [ ]:
plot_objective_space(min_index)

In [ ]:
plot_constraint_space(min_index)

In [ ]:
# Load the result object to a file using pickle
with open(pickle_filename, 'rb') as f:
    result = pickle.load(f)

plot_convergence_gp(result, "img/convergence.png")
plot_objective_gp(result, "img/partial_dependece.png")
plot_evaluations_gp(result, "img/parameter_space.png")
plot_regret_gp(result, "img/cumulative_regret.png")

## For all weight combinations

In [ ]:
all_weight_results = []
min_indices = []
for weights in weight_combinations:
    # Construct the filename based on the current set of weights
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    output_filename = f'results/output_{filename_weights}.pkl'
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'

    print(f"--- Processing for weights: {weights} ---")
    
    # Check if the file exists before trying to load it
    if not os.path.exists(output_filename):
        print(f"File not found: {output_filename}. Skipping this combination.")
        continue

    # Load the data from the pickle file
    try:
        with open(pickle_filename, 'rb') as f:
            result = pickle.load(f)
        min_indices.append(np.argmin(result.func_vals))
        print(min_indices[-1], result.func_vals[min_indices[-1]])
        
        with open(output_filename, 'rb') as f:
            loaded_output_list = pickle.load(f)

        # Extract the last element of each list for the desired statistics
        norm_emit_x = [stats['norm_emit_x'][-1] for stats in loaded_output_list]
        norm_emit_y = [stats['norm_emit_y'][-1] for stats in loaded_output_list]
        sigma_energy = [stats['sigma_energy'][-1] for stats in loaded_output_list]

        sigma_x = [stats['sigma_x'][-1] for stats in loaded_output_list]
        sigma_xp = [stats['sigma_xp'][-1] for stats in loaded_output_list]
        sigma_y = [stats['sigma_y'][-1] for stats in loaded_output_list]
        sigma_yp = [stats['sigma_yp'][-1] for stats in loaded_output_list]
        sigma_z = [stats['sigma_z'][-1] for stats in loaded_output_list]
        mean_kinetic_energy = [stats['mean_kinetic_energy'][-1] for stats in loaded_output_list]

        # Store the extracted results in a dictionary and append to the main list
        results = {
            'weights': weights,
            'norm_emit_x': norm_emit_x,
            'norm_emit_y': norm_emit_y,
            'sigma_energy': sigma_energy,
            'sigma_x': sigma_x,
            'sigma_xp': sigma_xp,
            'sigma_y': sigma_y,
            'sigma_yp': sigma_yp,
            'sigma_z': sigma_z,
            'mean_kinetic_energy': mean_kinetic_energy,
        }
        all_weight_results.append(results)
        
        print(f"Successfully loaded and extracted data for {len(loaded_output_list)} runs.")
    
    except Exception as e:
        print(f"An error occurred while processing {output_filename}: {e}")

print("\n--- All Data Processed ---")
print(f"Total number of weight combinations processed: {len(all_weight_results)}")

In [ ]:
norm_emit_x = [stats['norm_emit_x'] for stats in all_weight_results]
norm_emit_y = [stats['norm_emit_y'] for stats in all_weight_results]
sigma_energy = [stats['sigma_energy'] for stats in all_weight_results]

sigma_x = [stats['sigma_x'] for stats in all_weight_results]
sigma_xp = [stats['sigma_xp'] for stats in all_weight_results]
sigma_y = [stats['sigma_y'] for stats in all_weight_results]
sigma_yp = [stats['sigma_yp'] for stats in all_weight_results]
sigma_z = [stats['sigma_z'] for stats in all_weight_results]
mean_kinetic_energy = [stats['mean_kinetic_energy'] for stats in all_weight_results]

In [ ]:
plot_objective_space_all_iter('test.png')

In [ ]:
for i in range(0, len(weight_combinations)):
    print (weight_combinations[i], norm_emit_x[i][-1], norm_emit_y[i][-1], sigma_energy[i][-1])

In [ ]:
plot_objective_space_best("img/objective_space_best.png")
plot_objective_space_all("img/objective_space_all.png")
plot_constraint_space_all("img/constraint_space_all.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = cm.viridis(np.linspace(0.25, 1.0, len(weight_combinations)))

index = 0

for weights in weight_combinations:
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'

    with open(pickle_filename, 'rb') as f:
        result = pickle.load(f)

    plot_convergence_all(result, cc=colors[index], label=weights)

    index += 1

ax.set_title("Convergence Plot")
ax.set_xlabel("Number of Iterations $n$")
ax.set_ylabel("Objective Function Value")
ax.legend()
ax.grid()
fig.savefig("img/convergence_all.png")

In [ ]:
results = []

for weights in weight_combinations:
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'

    with open(pickle_filename, 'rb') as f:
        result = pickle.load(f)
    results.append(result)

filename = "img/convergence_gp.png"
plot_convergence_gp(results, filename)
filename = "img/regret_gp.png"
plot_regret_gp(results, filename)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

index = 0

for weights in weight_combinations:
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'

    with open(pickle_filename, 'rb') as f:
        result = pickle.load(f)

    plot_regret_all(result, cc=colors[index], label=weights)
    
    index += 1
    
ax.set_title("Cumulative Regret Plot")
ax.set_xlabel("Number of Iterations $n$")
ax.set_ylabel(r"$\sum_{i=0}^n(f(x_i) - optimum)$ after $n$ Iterations")
ax.legend()
ax.grid()
fig.savefig("img/cumulative_regret_all.png")